# Wartha Sensei — Local worker on Colab

Run the AI Local model on a free Colab GPU instead of Kaggle. The worker polls your backend (no ngrok) and uses the same model from your Kaggle dataset.

## Steps

1. In the app, open **AI Agent → Local** and tap **Run on Colab instead** to get a pairing code (valid 15 minutes, one use).
2. In Colab: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**.
3. Fill in the form above: **PAIR_CODE** (from the app), **KAGGLE_USERNAME** + **KAGGLE_KEY** (kaggle.com → Settings → API → Create New Token).
4. Run the cell. It claims the pairing, downloads the model dataset from Kaggle (~5–6 GB, once per session), then starts the worker. Leave the cell running — closing it stops the worker.
5. Back in the app, the top bar shows **Colab · Ready**. Stop it from the app (Stop Worker) as usual.


In [ ]:
# @title Run Wartha Sensei Worker { display-mode: "form" }
# Pairing code from the app (AI Agent -> Local -> Run on Colab):
PAIR_CODE = ""  # @param {type:"string"}
# Kaggle API token (kaggle.com -> Settings -> API -> Create New Token):
KAGGLE_USERNAME = ""  # @param {type:"string"}
KAGGLE_KEY = ""  # @param {type:"string"}
# Model context in tokens (32768 recommended on Colab T4 16 GB):
MAX_CTX = 32768  # @param {type:"integer"}

import glob
import json
import os
import shutil
import subprocess
import sys
import urllib.request

BACKEND_URL = "https://warthadev-backend.onrender.com"
DATASET_SLUG = "herbanuwarthakusuma/media-ai-local"
WORK_DIR = "/content/wartha"
DATASET_DIR = "/content/wartha-dataset"

# 0. GPU check - Colab runtime must be GPU (Runtime -> Change runtime type -> T4 GPU).
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU detected. In Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then run this cell again.")
print(gpu.stdout.strip())

if not PAIR_CODE.strip() or not KAGGLE_USERNAME.strip() or not KAGGLE_KEY.strip():
    raise SystemExit("Fill in PAIR_CODE, KAGGLE_USERNAME and KAGGLE_KEY above, then run this cell again.")

# 1. Claim the pairing code -> session credentials.
req = urllib.request.Request(
    f"{BACKEND_URL}/agent/local/colab-claim",
    data=json.dumps({"code": PAIR_CODE.strip().upper()}).encode(),
    headers={"Content-Type": "application/json"},
)
try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        claim = json.load(resp)
except Exception as exc:
    raise SystemExit(f"Pairing failed ({exc}). Check the code in the app (Local tab -> Run on Colab) - it expires in 15 minutes and works once.")
session_id = claim["session_id"]
worker_token = claim["worker_token"]
backend_url = claim["backend_url"]
print("Paired. Session:", session_id)

# 2. Model dataset from Kaggle (same GGUF the Kaggle worker uses).
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME.strip(), "key": KAGGLE_KEY.strip()}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
if not shutil.which("kaggle"):
    raise SystemExit("The kaggle command was not found after install. Restart the runtime and run this cell again.")
if not glob.glob(f"{DATASET_DIR}/**/model_gguf", recursive=True):
    print("Downloading model dataset from Kaggle (about 5-6 GB, once per session)...")
    subprocess.run(["kaggle", "datasets", "download", DATASET_SLUG, "-p", DATASET_DIR, "--unzip"], check=True)
else:
    print("Model dataset already downloaded, skipping.")

# 3. Worker code + start (polls the backend, no ngrok).
os.makedirs(WORK_DIR, exist_ok=True)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/warthadev/warthadev-backend/main/workerlocal.py",
    os.path.join(WORK_DIR, "workerlocal.py"),
)
sys.path.insert(0, WORK_DIR)
os.environ["WARTHA_BASE_DIR"] = WORK_DIR
os.environ["WARTHA_DATASET_DIR"] = DATASET_DIR
os.environ["WARTHA_MAX_CTX"] = str(MAX_CTX)
os.environ["WARTHA_CUDA_DEVICES"] = "0"
import workerlocal
print("Starting worker (Colab)...")
workerlocal.run_kaggle_worker(session_id=session_id, backend_url=backend_url, worker_token=worker_token)
